# EDA Full Dataset assembly

This notebook joins the six cleaned EDA splits using the shared `encounter_id` key. It preserves all 101,766 validated encounters and saves the final modeling-ready file as `EDA_Dataset/EDA_Full_Dataset.csv`.

## Imports and source files

In [1]:
from pathlib import Path  # Work with project file paths
import pandas as pd  # Load and validate CSV data

In [2]:
output_dir = Path("EDA_Dataset")  # Set the folder containing the cleaned splits
source_files = [output_dir / f"dataset_{number}.csv" for number in range(1, 7)]  # Build the six cleaned CSV paths

In [3]:
cleaned_splits = {path.stem: pd.read_csv(path, low_memory=False) for path in source_files}  # Load every cleaned split by filename

In [4]:
pd.DataFrame({name: frame.shape for name, frame in cleaned_splits.items()}, index=["rows", "columns"]).T  # Display the shape of each cleaned split

,rows,columns
dataset_1,101766,11
dataset_2,101766,10
dataset_3,101766,10
dataset_4,101766,10
dataset_5,101766,11
dataset_6,101766,9


## Validate keys and columns before joining

In [5]:
for dataset_name, frame in cleaned_splits.items():  # Validate every split before joining
    assert frame["encounter_id"].notna().all(), f"{dataset_name} has a missing encounter_id"  # Confirm the join key is complete
    assert frame["encounter_id"].is_unique, f"{dataset_name} has duplicate encounter_id values"  # Confirm one row per encounter
    assert frame.columns.is_unique, f"{dataset_name} has duplicate column names"  # Confirm unique feature names

In [6]:
reference_ids = cleaned_splits["dataset_1"]["encounter_id"]  # Use dataset 1 as the encounter-key reference
for dataset_name, frame in cleaned_splits.items():  # Compare each split with the reference key sequence
    assert frame["encounter_id"].tolist() == reference_ids.tolist(), f"{dataset_name} does not align with dataset_1"  # Confirm exact key alignment

In [7]:
constant_columns_by_split = {  # Collect any columns with exactly one observed value
    dataset_name: [column for column in frame.columns if column != "encounter_id" and frame[column].nunique(dropna=False) == 1]  # Identify zero-variance features
    for dataset_name, frame in cleaned_splits.items()  # Repeat the check for every cleaned split
}  # Finish the constant-column dictionary
constant_columns_by_split  # Display constant columns found before the join

{'dataset_1': [],
 'dataset_2': [],
 'dataset_3': [],
 'dataset_4': [],
 'dataset_5': [],
 'dataset_6': []}

## Join the six cleaned splits

In [8]:
full_df = cleaned_splits["dataset_1"].copy()  # Start the joined dataframe with dataset 1
for dataset_name in ["dataset_2", "dataset_3", "dataset_4", "dataset_5", "dataset_6"]:  # Add the remaining splits one at a time
    full_df = full_df.merge(cleaned_splits[dataset_name], on="encounter_id", how="inner", validate="one_to_one")  # Join on the common key and enforce one-to-one matching

In [9]:
assert full_df.shape[0] == len(reference_ids), "The join changed the number of encounters"  # Confirm every encounter was retained
assert full_df["encounter_id"].is_unique, "The joined dataset has duplicate encounter_id values"  # Confirm the joined key stays unique
assert full_df.columns.is_unique, "The joined dataset has duplicate columns"  # Confirm the joined feature names are unique

## Final data-quality checks

In [10]:
full_df.shape  # Display the joined dataset dimensions

(101766, 56)

In [11]:
full_df.head()  # Display the first five joined records

,encounter_id,num_medications,metformin-pioglitazone,diabetesMed,metformin-rosiglitazone,diag_1,glimepiride,chlorpropamide,nateglinide,tolbutamide,...,A1Cresult_missing,medical_specialty_missing,diag_2,readmitted,number_diagnoses,admission_source_id,glipizide-metformin,number_outpatient,acetohexamide,diag_2_missing
0,2278392,1,No,No,No,250.83,No,No,No,No,...,True,False,Unknown,NO,1,1,No,0,No,True
1,149190,18,No,Yes,No,276,No,No,No,No,...,True,True,250.01,>30,9,7,No,0,No,False
2,64410,13,No,Yes,No,648,No,No,No,No,...,True,True,250,NO,6,7,No,2,No,False
3,500364,16,No,Yes,No,8,No,No,No,No,...,True,True,250.43,NO,7,7,No,0,No,False
4,16680,8,No,Yes,No,197,No,No,No,No,...,True,True,157,NO,5,7,No,0,No,False


In [12]:
full_df.info()  # Display joined-column names, data types, and non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 56 columns):
 #   Column                     Non-Null Count   Dtype
---  ------                     --------------   -----
 0   encounter_id               101766 non-null  int64
 1   num_medications            101766 non-null  int64
 2   metformin-pioglitazone     101766 non-null  str  
 3   diabetesMed                101766 non-null  str  
 4   metformin-rosiglitazone    101766 non-null  str  
 5   diag_1                     101766 non-null  str  
 6   glimepiride                101766 non-null  str  
 7   chlorpropamide             101766 non-null  str  
 8   nateglinide                101766 non-null  str  
 9   tolbutamide                101766 non-null  str  
 10  diag_1_missing             101766 non-null  bool 
 11  diag_3                     101766 non-null  str  
 12  num_procedures             101766 non-null  int64
 13  acarbose                   101766 non-null  str  
 14  time_in_hospita

In [13]:
full_df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")  # Display missing values by joined column

,missing_count
encounter_id,0
num_medications,0
metformin-pioglitazone,0
diabetesMed,0
metformin-rosiglitazone,0
diag_1,0
glimepiride,0
chlorpropamide,0
nateglinide,0
tolbutamide,0


In [14]:
constant_columns_after_join = [column for column in full_df.columns if column != "encounter_id" and full_df[column].nunique(dropna=False) == 1]  # Identify any constant feature left after joining
constant_columns_after_join  # Display the remaining zero-variance feature list

[]

## Save the final EDA dataset

In [15]:
full_output_path = output_dir / "EDA_Full_Dataset.csv"  # Set the final joined CSV path
full_df.to_csv(full_output_path, index=False)  # Save the modeling-ready joined dataset
print(f"Saved {full_df.shape[0]:,} rows and {full_df.shape[1]:,} columns to {full_output_path}")  # Report the saved dataset dimensions

Saved 101,766 rows and 56 columns to EDA_Dataset\EDA_Full_Dataset.csv


In [16]:
saved_full_df = pd.read_csv(full_output_path, low_memory=False)  # Reload the final CSV for verification
assert saved_full_df.shape == full_df.shape, "Saved full dataset shape does not match"  # Confirm the saved CSV dimensions
assert saved_full_df["encounter_id"].is_unique, "Saved full dataset has duplicate encounter_id values"  # Confirm the saved join key remains unique
saved_full_df.head()  # Display the first five reloaded records

,encounter_id,num_medications,metformin-pioglitazone,diabetesMed,metformin-rosiglitazone,diag_1,glimepiride,chlorpropamide,nateglinide,tolbutamide,...,A1Cresult_missing,medical_specialty_missing,diag_2,readmitted,number_diagnoses,admission_source_id,glipizide-metformin,number_outpatient,acetohexamide,diag_2_missing
0,2278392,1,No,No,No,250.83,No,No,No,No,...,True,False,Unknown,NO,1,1,No,0,No,True
1,149190,18,No,Yes,No,276,No,No,No,No,...,True,True,250.01,>30,9,7,No,0,No,False
2,64410,13,No,Yes,No,648,No,No,No,No,...,True,True,250,NO,6,7,No,2,No,False
3,500364,16,No,Yes,No,8,No,No,No,No,...,True,True,250.43,NO,7,7,No,0,No,False
4,16680,8,No,Yes,No,197,No,No,No,No,...,True,True,157,NO,5,7,No,0,No,False
